# Delay state-space to state-space (dss2ss)

This example converts an FDN in **delay state-space** form (separate delay lengths and feedback matrix) into a single **state-space** system, and checks that the impulse response matches the delay-state-space implementation.

**What it does:**
- Builds a small lossless FDN: random orthogonal matrix, diagonal of gains `g^m`, and random input/output vectors.
- Uses `pyFDN.dss2ss` to get the equivalent state-space matrices `(A, b, c, d)`.
- Computes the impulse response both via `scipy.signal` (from the state-space) and via `pyFDN.dss2impz` (from the delay state-space).
- Plots both (mu-law encoded) and asserts they match within tolerance.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import ss2tf, dlti, dimpulse
import pyFDN

np.random.seed(1)
# Impulse response length for comparison
impulse_response_length = 100

# Lossless FDN: N delay lines, delay lengths m, gain g, random orthogonal feedback
N = 3
m = np.array([13, 19, 23])
g = 0.9
A = pyFDN.random_orthogonal(N) @ np.diag(g**m)
b = np.random.randn(N, 1)
c = np.random.randn(1, N)
d = np.random.randn(1, 1)

# Convert delay state-space to single state-space system
aa, bb, cc, dd = pyFDN.dss_to_ss(m, A, b, c, d)
# Via state-space (scipy): transfer function then impulse
num, den = ss2tf(aa, bb, cc, dd)
system = dlti(num, den)
_, ir_state_space = dimpulse(system, n=impulse_response_length)
ir_state_space = np.squeeze(ir_state_space)

# Via delay state-space (pyFDN)
ir_delay_state_space = pyFDN.dss_to_impz(
    impulse_response_length, m, A, b, c, d, input_type="mergeInput"
)

# Plot both (mu-law encoded; delay-state-space shifted for visibility)
plt.figure()
plt.plot(pyFDN.mulaw_encode(ir_state_space), label="State space")
plt.plot(pyFDN.mulaw_encode(ir_delay_state_space) + 2, label="Delay state space")
plt.xlabel("Time [samples]")
plt.ylabel("Amplitude [mu-law]")
plt.grid(True)
plt.legend()
plt.show()

# Sanity check: both implementations match
assert pyFDN.is_almost_zero(ir_state_space - ir_delay_state_space, tol=0.001)